## Compare pretrained backbones

In [7]:
from sklearn.model_selection import StratifiedGroupKFold
from configs.cfg import CFG
from dataset.biomass_dataset import *
from utils.augs import *
from configs.deterministic import *
from models.models import *
from train.train import *
from dataset.preprocess_data import *

if __name__ == "__main__":
    set_seed(CFG.SEED,CFG.DETERMINISTIC)
    df_wide = get_df()
    
    fold_indices = {}

    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=204)
    splitter = sgkf.split(X=df_wide, y=df_wide['biomass_bin'], groups=df_wide['group'])

    for fold_id, (train_idx, val_idx) in enumerate(splitter):
        fold_indices[fold_id] = val_idx
        print(f"Fold {fold_id} captured: {len(val_idx)} images")

    # A. Define which folds go where
    train_folds = [1, 2, 3, 4]
    val_fold    = 0
    test_fold   = 4

    # B. Construct the final index lists
    # np.concatenate joins the arrays of indices together
    train_idx_final = np.concatenate([fold_indices[f] for f in train_folds])
    val_idx_final   = fold_indices[val_fold]
    test_idx_final  = fold_indices[test_fold]
    # C. Create the DataFrames
    train_df = df_wide.iloc[train_idx_final].reset_index(drop=True)
    val_df   = df_wide.iloc[val_idx_final].reset_index(drop=True)
    # test_df  = df_wide.iloc[test_idx_final].reset_index(drop=True)
    # print(df_wide.head())
    print(f"Train Size: {len(train_df)}") # Should be roughly 60%
    print(f"Val Size:   {len(val_df)}")   # Should be roughly 20%
    # print(f"Test Size:  {len(test_df)}")  # Should be roughly 20%
    train_labels_tensor = torch.tensor(
        train_df[["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]].values, 
        dtype=torch.float32
    )
    # print(calculate_deltas(train_labels_tensor))
    group_name = f"Date_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    # best_r2 = train_base(train_df,val_df, 0,group_name=group_name)
# Epoch 100 | TrainLoss 71.63072 | ValLoss 1655.10278 | ValR² -0.1487 (BEST)

Loading data...
357 training images
Fold 0 captured: 72 images
Fold 1 captured: 75 images
Fold 2 captured: 87 images
Fold 3 captured: 65 images
Fold 4 captured: 58 images
Train Size: 285
Val Size:   72


In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()
del model, train_loader, val_loader, optimizer, main_scheduler

## Cross Validation Training

In [ ]:
import pandas as pd
import gc
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold
from configs.cfg import CFG
from torch.amp import GradScaler
from dataset.biomass_dataset import *
from utils.augs import *
from configs.deterministic import *
from models.models import *
from train.train import *
from dataset.preprocess_data import *
from utils.utils import *

if __name__ == "__main__":
    set_seed(CFG.SEED, CFG.DETERMINISTIC)
    df_wide = get_df()
    # best_seed = find_best_seed(df_wide, 1000)
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    splitter = sgkf.split(X=df_wide, y=df_wide['biomass_bin'], groups=df_wide['group'])
    # folds_list = list(splitter)

    # check_splits(splitter, df_wide)
    # Store the best R2 score from each fold
    all_fold_best_scores = []
    group_name = f"Date_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    for fold, (tr_idx, val_idx) in enumerate(splitter):
        print('\n' + '='*70)
        print(f'   FOLD {fold+1}/{CFG.N_FOLDS}   |   {len(tr_idx)} train / {len(val_idx)} val')
        print('='*70)
        fold_dir_512 = os.path.join('cached_folds_date_state_20_512', f"fold_{fold+1}")
        fold_dir_768 = os.path.join('cached_folds_date_state_20', f"fold_{fold+1}")
        fold_dir_1008 = os.path.join('cached_folds_date_state_20_1008', f"fold_{fold+1}")

        tr_df  = df_wide.iloc[tr_idx].reset_index(drop=True)
        val_df = df_wide.iloc[val_idx].reset_index(drop=True)

        best_r2 = train_base(fold_dir_768,tr_df,val_df,fold,group_name = group_name)
        # best_r2 = train_tree(fold_dir_768, fold, verbose=True)
        # best_r2 = train_knn(fold_dir_768, fold, verbose=True)
        
        all_fold_best_scores.append(best_r2)

    # Cleanup
    torch.cuda.empty_cache()
    gc.collect()
    # all_fold_sizes = [83,81,66,71,56]# Change by hand if folds change date
    all_fold_sizes = [76,74,90,57,60]# Change by hand if folds change state+date
    weighted_cv_score = np.average(all_fold_best_scores, weights=all_fold_sizes)

    # 2. Standard Deviation
    # For std, a simple np.std is usually fine to just show "stability," 
    # but you can stick to the simple calculation for this.
    std_cv_score = np.std(all_fold_best_scores)
    print('\n' + '='*70)
    print('         FINAL CROSS-VALIDATION SCORE')
    print('='*70)
    print(f'Public LB Score: 0.58')
    # Notice we use the weighted score here
    print(f'Local CV Score: {weighted_cv_score:.4f} ± {std_cv_score:.4f}')
    print('\nIndividual Fold Scores:')
    for i, (score, size) in enumerate(zip(all_fold_best_scores, all_fold_sizes)):
        print(f'  Fold {i+1} (n={size}): {score:.4f}')
    # Local CV Score: 0.8078 ± 0.0415
    # 0.7953 ± 0.0358 - mse
    # 512,mse: 0.7993 ± 0.0364
    # 768,mse: 0.7933 ± 0.0370
    # 1008,mse: 0.7899 ± 0.0413


/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...
357 training images

   FOLD 1/5   |   281 train / 76 val


NameError: name 'best_r2' is not defined